<a href="https://colab.research.google.com/github/spicecat/unhash/blob/main/numba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[https://scratch.mit.edu/projects/164028530](https://scratch.mit.edu/projects/164028530)

In [1]:
# @title Register Ruff Cell Magic
%pip install ruff -q

from IPython.core.magic import register_cell_magic
import subprocess
import tempfile
import os


@register_cell_magic
def ruff_format(line, cell):
    """Cell magic to format code with ruff. Usage: %%ruff_format"""
    with tempfile.NamedTemporaryFile(mode="w+", delete=False, suffix=".py") as f:
        f.write(cell)
        file_path = f.name
    try:
        subprocess.run(
            ["ruff", "format", file_path], check=True, capture_output=True, text=True
        )
        with open(file_path, "r") as f:
            print(f.read())
    except subprocess.CalledProcessError as e:
        print("Error during formatting:", e.stderr)
    finally:
        os.remove(file_path)


@register_cell_magic
def ruff_lint(line, cell):
    """Cell magic to lint code with ruff. Usage: %%ruff_lint"""
    with tempfile.NamedTemporaryFile(mode="w+", delete=False, suffix=".py") as f:
        f.write(cell)
        file_path = f.name
    try:
        result = subprocess.run(
            ["ruff", "check", file_path], capture_output=True, text=True
        )
        if result.stdout:
            print(result.stdout)
        else:
            print("No issues found.")
    finally:
        os.remove(file_path)


# Numba

In [2]:
# @title Import libraries

import numpy as np
from numpy.typing import NDArray
import numba as nb

np.set_printoptions(formatter={"int": hex})


## Hash

### encode

In [3]:
# @title Define encode

Char = np.uint8
NChar = nb.types.uint8
Enc = NDArray[Char]

H = Char(2)
HRange = np.arange(H, dtype=Char)
CW = Char(np.iinfo(Char).bits // H)
C = 1 << CW
CHARS = "abcdefghijklmnopqrstuvwxyz0123456789 "[:C]


@nb.jit(NChar[:](nb.types.string))
def encode(s: str) -> Enc:
    enc = np.array([+CHARS.index(c) for c in s], dtype=Char)
    return (enc.reshape(-1, H) << CW * HRange).sum(axis=1, dtype=Char)


@nb.jit(nb.types.string(NChar[:]))
def decode(e: Enc) -> str:
    enc = (e.repeat(H).reshape(-1, H) >> CW * HRange & C - 1).flatten()
    return "".join([CHARS[c] for c in enc])


del HRange, CW, CHARS


In [ ]:
# @title Test encode

s = "apabcdef"
print(s, encode(s))
assert decode(encode(s)) == s
del s


### hashify

In [4]:
# @title Define hashify

Con = np.uint16
NCon = nb.types.uint16
Cons = NDArray[Con]

MQ = np.float64(np.iinfo(Con).max + 1)
M = np.uint64(100_000)
Q = MQ / M

S = Char(4)
N = Char(16)
assert N % H == 0
P0 = np.array([11, 17, 7, 5], np.uint32)
P1 = np.array([29, 31, 17, 13], np.uint32)
P2 = np.array([53, 67, 103, 47], np.uint32)
P3 = np.array([52, 12, 24, 30], np.uint32)
P4 = np.array([0, 90, 0, 90], np.uint32)
idx = np.arange(N, dtype=np.uint32)
E = np.outer(np.arange(C, dtype=Char), np.ones(N, dtype=Char))[:, :, np.newaxis]
sin = 5 * np.sin(np.radians((np.outer(idx + 1, P0) % P1 + P2) * (E + P3) + P4))
CTABLE = np.round((sin - np.floor(sin)) * MQ).astype(Con)
HTABLE: Cons = (
    CTABLE.reshape(C, N // H, H, S)
    .transpose(2, 0, 1, 3)[
        np.arange(H, dtype=Char).reshape((H,) + (1,) * H),
        np.indices((C.tolist(),) * H)[::-1],
    ]
    .sum(axis=0, dtype=Con)
    .reshape(-1, N // H, S)
)


@nb.guvectorize([(NChar[:], NCon[:, :, :], NCon[:])], "(l),(c,n,s)->(s)")
def hashify(enc: Enc, htable: Cons, cons: Cons):
    for i in range(len(enc)):
        cons += htable[enc[i], i]


del MQ, M, P0, P1, P2, P3, P4, idx, E, sin, CTABLE


In [ ]:
# @title Test hashify

s = ["apabcdef", "mkkaphmi", "pckkemhj", "fhgkgbhl"]
enc = np.array(list(map(encode, s)))
cons = np.zeros((len(enc), S), dtype=Con)
hashify(enc, HTABLE, cons)
print("s=", s, "enc=", enc, "cons=", cons, sep="\n")
assert np.array_equal(
    cons,
    np.array(
        [
            [0xB88C, 0x8BD4, 0xEF8E, 0x9F26],
            [0x499B, 0x301E, 0x2BCA, 0xC5BD],
            [0xED63, 0x93AE, 0xA66, 0xD9A5],
            [0x7DA1, 0x30CC, 0xC30, 0x5552],
        ],
        dtype=Con,
    ),
)
del s, enc, cons


In [ ]:
# @title Time hashify

seed = 0  # @param {"type":"integer","placeholder":"0"}
n_keys = 100_000_000  # @param {"type":"integer","placeholder":"100_000_000"}

np.random.seed(seed)
W = 4
enc = np.random.randint(0, np.iinfo(Char).max, n_keys * W, dtype=Char).reshape(-1, W)

cons = np.empty((n_keys, S), dtype=Con)
%timeit hashify(enc, HTABLE, cons)
del seed, n_keys, W, enc, cons
# @markdown 3.57 s ± 1.02 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Binary fuse filter

### pack

In [5]:
# @title Define pack

Key = np.uint64
Keys = NDArray[Key]
NKey = nb.types.uint64

ERR = Con(N // 2)
MASK = ~(1 << ERR) + 1

@nb.guvectorize([(NCon[:], NKey[:])], "(s)->()")
def pack(cons: Cons, key: Keys):
    temp_key = Key(0)
    for i in range(S):
        temp_key |= (cons[i] & MASK) << i * np.iinfo(Key).bits // S
    key[0] = temp_key


del ERR, MASK

# ERR = Con(N // 2.5)
# L = np.iinfo(Con).bits
# SRange = np.arange(S, dtype=np.uint16)
# SW = np.iinfo(Key).bits // Key(S)
# SShift = (S - 1) * SW - SRange * L % np.iinfo(Key).bits


# @nb.guvectorize([(NCon[:], NKey[:])], "(s)->()")
# def pack(cons: Cons, key: Keys):
#     key[0] = (
#         (
#             (cons >> ERR).view(Key)[SRange * L // np.iinfo(Key).bits] >> SShift
#             & Key((1 << SW) - 1)
#         )
#         << SW * SRange
#     ).sum(dtype=Key)


# del ERR, L, SRange, SW, SShift


In [ ]:
# @title Test pack

s = ["apabcdef", "mkkaphmi", "pckkemhj", "fhgkgbhl"]
enc = np.array(list(map(encode, s)))
cons = np.zeros((len(enc), S), dtype=Con)
hashify(enc, HTABLE, cons)
keys = np.zeros(len(s), dtype=Key)
pack(cons, keys)

target = ((
    np.array(
        [
            [72090, 54618, 93575, 62167],
            [28752, 18794, 17102, 77239],
            [92729, 57686, 4061, 85015],
            [49071, 19061, 4760, 33326],
        ],
        dtype=Key,
    )
    * Q
)).astype(Con)
target_keys = np.zeros(len(target), dtype=Key)
pack(target, target_keys)

print(
    "s=",
    s,
    "enc=",
    enc,
    "cons=",
    cons,
    "target=",
    target,
    "keys=",
    keys,
    "target_keys=",
    target_keys,
    "diff=",
    keys - target_keys,
    sep="\n",
)
assert np.array_equal(keys, target_keys)
del s, enc, cons, target, keys, target_keys


In [ ]:
# @title Time pack

seed = 0 # @param {"type":"integer","placeholder":"0"}
n_keys = 100_000_000 # @param {"type":"integer","placeholder":"100_000_000"}

np.random.seed(seed)
W = 4
enc = np.random.randint(0, np.iinfo(Char).max, n_keys * W, dtype=Char).reshape(-1, W)

cons = np.empty((n_keys, S), dtype=Con)
hashify(enc, HTABLE, cons)
keys = np.empty(n_keys, dtype=Key)
%timeit pack(cons, keys)
del seed, n_keys, W, enc, cons, keys
# @markdown 497 ms ± 8.45 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### populate

In [6]:
# @title Define BinaryFuseFilter

FPrint = np.uint16
NFPrint = nb.types.uint16
FPrints = NDArray[FPrint]

@nb.jit(NKey(NKey, NKey))
def mulhi(a: Key, b: Key) -> Key:
    a_low = a & Key(0xFFFFFFFF)
    a_high = a >> Key(32)
    b_low = b & Key(0xFFFFFFFF)
    b_high = b >> Key(32)
    p00 = a_low * b_low
    p01 = a_low * b_high
    p10 = a_high * b_low
    p11 = a_high * b_high
    middle = p10 + (p00 >> Key(32)) + (p01 & Key(0xFFFFFFFF))
    return p11 + (middle >> Key(32)) + (p01 >> Key(32))

@nb.jit(NFPrint(NKey))
def fingerprint(key: Key) -> FPrint:
    return FPrint(key)


spec = [
    ("n_keys", nb.types.uint64),
    ("segment_length", nb.types.uint64),
    ("segment_length_mask", nb.types.uint64),
    ("total_segments", nb.types.uint64),
    ("segment_count", nb.types.uint64),
    ("segment_count_length", nb.types.uint64),
    ("fingerprints", NFPrint[:]),
]


@nb.experimental.jitclass(spec)
class BinaryFuseFilter:
    def __init__(self, n_keys):
        self.n_keys = n_keys

        ARITY = np.uint64(3)
        self.segment_length = np.uint64(1) << np.uint64(
            np.log2(n_keys) / np.log2(3.33) + 2.25
        )
        self.segment_length_mask = self.segment_length - np.uint64(1)
        self.total_segments = (
            np.uint64(n_keys * 1.125) + self.segment_length_mask
        ) // self.segment_length
        self.segment_count = self.total_segments - (ARITY - np.uint64(1))
        self.segment_count_length = self.segment_count * self.segment_length

    def _get_hashes(self, key: Key):
        h0 = mulhi(key, self.segment_count_length)
        h1 = h0 + self.segment_length ^ (key >> Key(18)) & self.segment_length_mask
        h2 = (
            h0 + self.segment_length + self.segment_length
            ^ key & self.segment_length_mask
        )
        return h0, h1, h2

    def populate(self, keys: Keys):
        block_bits = 1
        while (1 << block_bits) < self.segment_count:
            block_bits += 1
        block = 1 << block_bits
        start_pos = np.zeros(block, dtype=np.uint64)
        for i in range(block):
            start_pos[i] = np.uint64((i * n_keys) >> block_bits)

        reverse_order = np.zeros(n_keys + 1, dtype=Key)
        reverse_order.fill(0)
        reverse_order[n_keys] = 1
        mask_block = Key(block - 1)
        for h in keys:
            segment_index = h >> (64 - block_bits)
            while reverse_order[start_pos[segment_index]] != 0:
                segment_index = (segment_index + Key(1)) & mask_block
            reverse_order[start_pos[segment_index]] = h
            start_pos[segment_index] += 1

        capacity = self.total_segments * self.segment_length
        t2count = np.zeros(capacity, dtype=np.uint8)
        t2hash = np.zeros(capacity, dtype=Key)
        for i in range(n_keys):
            hash_val = reverse_order[i]
            h0, h1, h2 = self._get_hashes(hash_val)
            t2count[h0] += 4
            t2hash[h0] ^= hash_val
            t2count[h1] += 4
            t2count[h1] ^= 1
            t2hash[h1] ^= hash_val
            t2count[h2] += 4
            t2count[h2] ^= 2
            t2hash[h2] ^= hash_val

        alone = np.zeros(capacity, dtype=np.uint64)
        q_size = 0
        for i in range(capacity):
            if (t2count[i] >> 2) == 1:
                alone[q_size] = i
                q_size += 1

        reverse_h = np.zeros(n_keys, dtype=np.uint8)
        stack_size = 0
        while q_size > 0:
            q_size -= 1
            index = alone[q_size]
            if (t2count[index] >> 2) == 1:
                hash_val = Key(t2hash[index])
                found = np.uint8(t2count[index] & 3)
                reverse_h[stack_size] = found
                reverse_order[stack_size] = hash_val
                stack_size += 1
                h_all = self._get_hashes(hash_val)

                other_index1 = h_all[(found + 1) % 3]
                if (t2count[other_index1] >> 2) == 2:
                    alone[q_size] = other_index1
                    q_size += 1
                t2count[other_index1] -= 4
                t2count[other_index1] ^= (found + 1) % 3
                t2hash[other_index1] ^= hash_val

                other_index2 = h_all[(found + 2) % 3]
                if (t2count[other_index2] >> 2) == 2:
                    alone[q_size] = other_index2
                    q_size += 1
                t2count[other_index2] -= 4
                t2count[other_index2] ^= (found + 2) % 3
                t2hash[other_index2] ^= hash_val

        self.fingerprints = np.zeros(capacity, dtype=FPrint)
        for i in range(n_keys - 1, -1, -1):
            hash_val = reverse_order[i]
            found = reverse_h[i]
            h_all = self._get_hashes(hash_val)
            self.fingerprints[h_all[found]] = (
                fingerprint(hash_val)
                ^ self.fingerprints[h_all[(found + 1) % 3]]
                ^ self.fingerprints[h_all[(found + 2) % 3]]
            )

    def contain(self, key: Key) -> np.bool:
        h0, h1, h2 = self._get_hashes(key)
        return (
            fingerprint(key)
            ^ self.fingerprints[h0]
            ^ self.fingerprints[h1]
            ^ self.fingerprints[h2]
            == 0
        )

    def contains(self, keys: Keys) -> NDArray[np.bool]:
        contain = np.empty(len(keys), dtype=np.bool)
        for i, key in enumerate(keys):
            contain[i] = self.contain(key)
        return contain


In [16]:
# @title Test populate

seed = 0  # @param {"type":"integer","placeholder":"0"}
n_keys = 2_000_000  # @param {"type":"integer","placeholder":"2_000_000"}

np.random.seed(seed)
keys = np.random.randint(0, np.iinfo(Key).max, n_keys, dtype=Key)
assert np.unique(keys).size == n_keys

bff = BinaryFuseFilter(n_keys)
bff.populate(keys)
print(FPrint, len(bff.fingerprints), len(keys))
del seed, bff


<class 'numpy.uint16'> 2260992 2000000


In [17]:
# @title Time populate

seed = 0  # @param {"type":"integer","placeholder":"0"}
n_keys = 2_000_000  # @param {"type":"integer","placeholder":"2_000_000"}

np.random.seed(seed)
keys = np.random.randint(0, np.iinfo(Key).max, n_keys, dtype=Key)
assert np.unique(keys).size == n_keys

bff = BinaryFuseFilter(n_keys)
%timeit bff.populate(keys)
del seed, bff

# @markdown 278 ms ± 11.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


278 ms ± 11.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
# @title Test contain

seed = 0  # @param {"type":"integer","placeholder":"0"}
n_keys = 2_000_000  # @param {"type":"integer","placeholder":"2_000_000"}

np.random.seed(seed)
keys = np.random.randint(0, np.iinfo(Key).max, n_keys, dtype=Key)
assert np.unique(keys).size == n_keys

bff = BinaryFuseFilter(n_keys)
bff.populate(keys)

true_positives = bff.contains(keys).sum()
print(f"{n_keys=} {true_positives=}")
assert true_positives == n_keys

test_seed = 1  # @param {"type":"integer","placeholder":"1"}
n_test_keys = 10_000_0000  # @param {"type":"integer","placeholder":"100_000_000"}

np.random.seed(test_seed)
test_keys = np.random.randint(0, np.iinfo(Key).max, size=n_test_keys, dtype=Key)
false_positives = bff.contains(test_keys).sum()
expected_false_positives = n_test_keys / (1 << np.iinfo(FPrint).bits)
print(f"{false_positives=} {expected_false_positives=}")
assert false_positives < 2 * expected_false_positives
del (
    seed,
    n_keys,
    keys,
    bff,
    test_seed,
    n_test_keys,
    test_keys,
    true_positives,
    false_positives,
    expected_false_positives,
)


n_keys=2000000 true_positives=np.int64(2000000)
false_positives=np.int64(1617) expected_false_positives=1525.87890625


In [13]:
# @title Time contain

seed = 0  # @param {"type":"integer","placeholder":"0"}
n_keys = 2_000_000  # @param {"type":"integer","placeholder":"2_000_000"}

np.random.seed(seed)
keys = np.random.randint(0, np.iinfo(Key).max, n_keys, dtype=Key)
assert np.unique(keys).size == n_keys

bff = BinaryFuseFilter(n_keys)
bff.populate(keys)

test_seed = 1  # @param {"type":"integer","placeholder":"1"}
n_test_keys = 10_000_000  # @param {"type":"integer","placeholder":"10_000_000"}

np.random.seed(test_seed)
test_keys = np.random.randint(0, np.iinfo(Key).max, n_test_keys, dtype=Key)
%timeit bff.contains(test_keys)

del seed, n_keys, keys, bff, test_seed, n_test_keys, test_keys
# @markdown 132 ms ± 22.5 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


132 ms ± 22.5 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [18]:
# @title Test import

seed = 0  # @param {"type":"integer","placeholder":"0"}
n_keys = 2_000_000  # @param {"type":"integer","placeholder":"2_000_000"}

np.random.seed(seed)
keys = np.random.randint(0, np.iinfo(Key).max, n_keys, dtype=Key)
assert np.unique(keys).size == n_keys

bff = BinaryFuseFilter(n_keys)
bff.populate(keys)

np.save("fingerprints.npy", bff.fingerprints)
del seed, bff

fingerprints = np.load("fingerprints.npy")
bff = BinaryFuseFilter(n_keys)
bff.fingerprints = fingerprints

true_positives = bff.contains(keys).sum()
print(f"{n_keys=} {true_positives=}")
assert true_positives == n_keys

test_seed = 1  # @param {"type":"integer","placeholder":"1"}
n_test_keys = 10_000_0000  # @param {"type":"integer","placeholder":"100_000_000"}

np.random.seed(test_seed)
test_keys = np.random.randint(0, np.iinfo(Key).max, size=n_test_keys, dtype=Key)
false_positives = bff.contains(test_keys).sum()
expected_false_positives = n_test_keys / (1 << np.iinfo(FPrint).bits)
print(f"{false_positives=} {expected_false_positives=}")
assert false_positives < 2 * expected_false_positives
del (
    n_keys,
    keys,
    bff,
    true_positives,
    test_seed,
    n_test_keys,
    test_keys,
    false_positives,
    expected_false_positives,
)


n_keys=2000000 true_positives=np.int64(2000000)
false_positives=np.int64(1617) expected_false_positives=1525.87890625


In [74]:
# @title Define ShardedBinaryFuseFilter

class ShardedBinaryFuseFilter:
    def __init__(self, bffs):
        self.bffs = bffs

    def contains(self, keys: Keys) -> NDArray[np.bool]:
        contain = np.empty(len(keys), dtype=np.bool)
        for i in range(len(self.bffs)):
            ki = keys % len(self.bffs) == i
            contain[ki] = self.bffs[i].contains(keys[ki])
        return contain


In [75]:
# @title Test ShardedBinaryFuseFilter

seed = 0  # @param {"type":"integer","placeholder":"0"}
n_keys = 2_000_000  # @param {"type":"integer","placeholder":"2_000_000"}

np.random.seed(seed)
keys = np.random.randint(0, np.iinfo(Key).max, n_keys, dtype=Key)
assert np.unique(keys).size == n_keys

keys0 = keys[keys % 2 == 0]
n_keys0 = len(keys0)
keys1 = keys[keys % 2 == 1]
n_keys1 = len(keys1)
print(f"{n_keys=} {n_keys0=} {n_keys1=}")

bff0 = BinaryFuseFilter(n_keys0)
bff0.populate(keys0)
bff1 = BinaryFuseFilter(n_keys1)
bff1.populate(keys1)

sbff = ShardedBinaryFuseFilter(nb.typed.List([bff0, bff1]))

true_positives = sbff.contains(keys).sum()
print(f"{n_keys=} {true_positives=}")
assert true_positives == n_keys

test_seed = 1  # @param {"type":"integer","placeholder":"1"}
n_test_keys = 10_000_000  # @param {"type":"integer","placeholder":"10_000_000"}

np.random.seed(test_seed)
test_keys = np.random.randint(0, np.iinfo(Key).max, size=n_test_keys, dtype=Key)
false_positives = sbff.contains(test_keys).sum()
expected_false_positives = n_test_keys / (1 << np.iinfo(FPrint).bits)
print(f"{false_positives=} {expected_false_positives=}")
assert false_positives < 2 * expected_false_positives

del (
    seed,
    n_keys,
    keys,
    keys0,
    n_keys0,
    keys1,
    n_keys1,
    bff0,
    bff1,
    sbff,
    test_seed,
    n_test_keys,
    test_keys,
    true_positives,
    false_positives,
    expected_false_positives,
)


n_keys=2000000 n_keys0=1000909 n_keys1=999091
n_keys=2000000 true_positives=np.int64(2000000)
false_positives=np.int64(210) expected_false_positives=152.587890625


In [76]:
# @title Time ShardedBinaryFuseFilter

seed = 0  # @param {"type":"integer","placeholder":"0"}
n_keys = 2_000_000  # @param {"type":"integer","placeholder":"2_000_000"}

np.random.seed(seed)
keys = np.random.randint(0, np.iinfo(Key).max, n_keys, dtype=Key)
assert np.unique(keys).size == n_keys

keys0 = keys[keys % 2 == 0]
n_keys0 = len(keys0)
keys1 = keys[keys % 2 == 1]
n_keys1 = len(keys1)
print(f"{n_keys=} {n_keys0=} {n_keys1=}")

bff0 = BinaryFuseFilter(n_keys0)
bff0.populate(keys0)
bff1 = BinaryFuseFilter(n_keys1)
bff1.populate(keys1)

sbff = ShardedBinaryFuseFilter(nb.typed.List([bff0, bff1]))

test_seed = 1  # @param {"type":"integer","placeholder":"1"}
n_test_keys = 10_000_000  # @param {"type":"integer","placeholder":"10_000_000"}

np.random.seed(test_seed)
test_keys = np.random.randint(0, np.iinfo(Key).max, size=n_test_keys, dtype=Key)
%timeit sbff.contains(test_keys)
%timeit sbff.bffs[0].contains(test_keys[test_keys % 2 == 0])
%timeit sbff.bffs[1].contains(test_keys[test_keys % 2 == 1])

del (
    seed,
    n_keys,
    keys,
    keys0,
    n_keys0,
    keys1,
    n_keys1,
    bff0,
    bff1,
    sbff,
    test_seed,
    n_test_keys,
    test_keys,
)

# @markdown 1.17 s ± 203 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


n_keys=2000000 n_keys0=1000909 n_keys1=999091
939 ms ± 296 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
278 ms ± 8.02 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
279 ms ± 4.57 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Search

In [ ]:
# @title Populate Binary Fuse Filter

LEnc = np.uint32

ERR = Con(N // 2)
MASK = ~(1 << ERR) + 1
@nb.guvectorize([(NChar[:], NCon[:, :, :], NKey[:])], "(l),(c,n,s)->()")
def hashify_pack(enc: Enc, htable: Cons, key: Keys):
    cons = np.zeros(S, dtype=Con)
    for i in range(len(enc)):
        cons += htable[enc[i], i]
    temp_key = Key(0)
    for i in range(S):
        temp_key |= (cons[i] & MASK) << i * np.iinfo(Key).bits // S
    key[0] = temp_key
del ERR, MASK

gigabytes = 0.01  # @param {"type":"number","placeholder":"4.0"}

n_keys = int(gigabytes * 1024**3 // (np.dtype(LEnc).itemsize + np.dtype(Key).itemsize))
enc = np.arange(0, n_keys, dtype=LEnc).view(Char).reshape(n_keys, -1)
print(n_keys, enc)

keys = np.zeros(n_keys, dtype=Key)
hashify_pack(enc, HTABLE, keys)
print(keys)
del gigabytes, enc

bff = BinaryFuseFilter(n_keys)
bff.populate(keys)

del n_keys, keys


### search

In [ ]:
### Define search
L = 5

np.random.seed(42)
target = np.random.randint(0, C, L)


@nb.jit
def check(a: np.ndarray):
    return np.array_equal(a, target)


@nb.jit(parallel=True)
def search():
    done = False
    res = np.zeros(L, dtype=Con)
    for i in nb.prange(C):
        if done:
            continue
        enc = np.full(L, C - 1, dtype=Con)
        enc[0] = i
        while True:
            if check(enc):
                done = True
                for k in range(N):
                    res[k] = enc[k]
                break
            j = L - 1
            while j > 0:
                if enc[j]:
                    enc[j] -= 1
                    break
                else:
                    enc[j] = C - 1
                    j -= 1
            else:
                break
    return res


# search()


In [ ]:
%timeit search()

In [ ]:
import numpy as np
import numba as nb

@nb.njit(parallel=True)
def generate_first_m_combinations(M, N=32, L=6):
    mask = 31
    shift_amount = 4
    result = np.empty((M, L), dtype=np.int64)
    for i in nb.prange(M):
        val = i + 0
        for j in range(L):
            result[i, j] = val & mask
            val >>= shift_amount
    return result

N = 32
L = 6
M = 100_000_000
# first_combinations = generate_first_m_combinations(M, N, L)
# first_combinations.nbytes / (1 << 30)

In [ ]:
N = 32
L = 6
M = 100_000_000
%timeit generate_first_m_combinations(M, N, L)

In [ ]:
del Char, NChar, Enc, Con, NCon, Cons, S, HTABLE

# Numba CUDA

In [ ]:
# import numpy as np
# from numpy.typing import NDArray
# import numba as nb
# from numba import cuda

# np.set_printoptions(formatter={'int':hex})
# cuda.config.CUDA_ENABLE_PYNVJITLINK = True

### encode

In [ ]:
# @title Define encode

# Char = np.uint8
# NChar = nb.types.uint8
# Enc = NDArray[Char]

# H = Char(2)
# HRange = np.arange(H, dtype=Char)
# CW = Char(np.iinfo(Char).bits // H)
# C = 1 << CW
# CHARS = "abcdefghijklmnopqrstuvwxyz0123456789 "[:C]


# @nb.jit(NChar[:](nb.types.string))
# def encode(s: str) -> Enc:
#     enc = np.array([+CHARS.index(c) for c in s], dtype=Char)
#     return (enc.reshape(-1, H) << CW * HRange).sum(axis=1, dtype=Char)


# @nb.jit(nb.types.string(NChar[:]))
# def decode(e: Enc):
#     enc = (
#         e.repeat(H).reshape(-1, H) >> CW * HRange & C - 1
#     ).flatten()
#     return "".join([CHARS[c] for c in enc])


# del HRange, CW, CHARS

# s = "apabcdef"
# enc = encode(s)
# print(f"{H=} {C=} {enc} {decode(enc)}")
# del s, enc

## Hash

In [ ]:
# Con = np.uint16
# NCon = nb.types.uint16
# Cons = NDArray[Con]

# MQ = np.float64(np.iinfo(Con).max + 1)
# M = np.uint64(100_000)
# Q = MQ / M

# S = Char(4)
# N = Char(16)
# assert N % H == 0
# P0 = np.array([11, 17, 7, 5], np.uint64)
# P1 = np.array([29, 31, 17, 13], np.uint16)
# P2 = np.array([53, 67, 103, 47], np.uint16)
# P3 = np.array([52, 12, 24, 30], np.uint16)
# P4 = np.array([0, 90, 0, 90], np.uint64)
# idx = np.arange(N, dtype=np.uint64)
# A = np.outer(idx + 1, P0) % P1 + P2
# B = A * P3 + P4
# E = np.outer(np.arange(C, dtype=Char), np.ones(N, dtype=Char))[:, :, np.newaxis]
# angle = A * E + B
# sin = 5 * np.sin(np.radians(angle))
# CTABLE = np.round((sin - np.floor(sin)) * MQ).astype(Con)
# HTABLE: Cons = (
#     CTABLE.reshape(C, N // H, H, S)
#     .transpose(2, 0, 1, 3)[
#         np.arange(H, dtype=Char).reshape((H,) + (1,) * H),
#         np.indices((C.tolist(),) * H)[::-1],
#     ]
#     .sum(axis=0, dtype=Con)
#     .reshape(-1, N // H, S)
# )


# @cuda.jit([(NChar[:,:], NCon[:, :, :], NCon[:, :])])
# def cu_hashify(enc: Enc, htable: Cons, cons: Cons):
#     tid = cuda.grid(1)
#     if tid >= len(cons):
#         return
#     for i in range(enc.shape[1]):
#         for s in range(S):
#             cons[tid, s] += htable[enc[tid, i], i, s]


# del MQ, M, P0, P1, P2, P3, P4, idx, A, B, E, angle, sin, CTABLE

# s = ["apabcdef", "mkkaphmi", "pckkemhj", "fhgkgbhl"]
# enc = np.array(list(map(encode, s)))
# cu_enc = cuda.to_device(enc)
# cu_cons = cuda.to_device(np.zeros((len(enc), S), dtype=Con))
# cu_htable = cuda.to_device(HTABLE)
# cu_hashify.forall(len(s))(cu_enc, cu_htable, cu_cons)
# print(cu_cons.copy_to_host())
# del s, enc, cu_enc, cu_cons

In [ ]:
# W = 4
# n_keys = 100_000_000
# np.random.seed(0)
# enc = np.random.randint(0, np.iinfo(Char).max, n_keys * W, dtype=Char).reshape(-1, W)

# cu_enc = cuda.to_device(enc)
# cu_cons = cuda.to_device(np.zeros((n_keys, S), dtype=Con))
# cu_htable = cuda.to_device(HTABLE)
# %timeit cu_hashify.forall(NT)(cu_enc, cu_htable, cu_cons)

# del W, n_keys, enc, cu_enc, cu_cons, cu_htable
# # 164 µs ± 90.4 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
# Key = np.uint64
# Keys = NDArray[Key]
# NKey = nb.types.uint64

# ERR = Con(N // 2.5)

# @cuda.jit([(NCon[:, :], NKey[:])])
# def cu_pack(cons: Cons, key: Keys):
#     tid = cuda.grid(1)
#     temp_key = Key(0)
#     for i in range(4):
#         temp_key |= cons[tid, i] >> ERR << i * np.iinfo(Key).bits // S
#     key[tid] = temp_key

# s = ["apabcdef", "mkkaphmi", "pckkemhj", "fhgkgbhl"]
# enc = np.array(list(map(encode, s)))
# cons = np.empty((len(enc), S), dtype=Con)
# hashify(enc, HTABLE, cons)
# target = np.array(
#     [
#         [72090, 54618, 93575, 62167],
#         [28752, 18794, 17102, 77239],
#         [92729, 57686, 4061, 85015],
#         [49071, 19061, 4760, 33326]
#     ],
#     dtype=Key,
# )
# cu_cons = cuda.to_device(cons)
# cu_keys = cuda.to_device(np.empty(len(target), dtype=Key))
# cu_pack.forall(len(target))(cu_cons, cu_keys)
# print(cu_keys.copy_to_host())

# del s, enc, cons, target, cu_cons, cu_keys

In [ ]:
# # @title Time pack

# n_keys = 100_000_000 # @param {"type":"integer","placeholder":"100_000_000"}
# W = 4
# np.random.seed(0)
# enc = np.random.randint(0, np.iinfo(Char).max, n_keys * W, dtype=Char).reshape(-1, W)

# cons = np.empty((n_keys, S), dtype=Con)
# hashify(enc, HTABLE, cons)
# cu_cons = cuda.to_device(cons)
# cu_keys = cuda.to_device(np.zeros(n_keys, dtype=Key))
# %timeit cu_pack.forall(NT)(cu_cons, cu_keys)

# del W, n_keys, enc, cons, cu_cons, cu_keys
# # 6.45 ms ± 6.16 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


## Binary Fuse Filter

In [ ]:
# Key = np.uint64
# NKey = nb.types.uint64
# Keys = NDArray[Key]
# NKeys = NKey[:]

# fingerprints = np.load('fingerprints.npy')
# size = 1_000_000

# segment_length = 1 << int(np.log(size) / np.log(3.33) + 2.25)
# segment_length_mask = segment_length - 1

# ARITY = 3
# total_segments = (int(size * 1.125) + segment_length_mask) // segment_length
# segment_count = total_segments - (ARITY - 1)
# segment_count_length = segment_count * segment_length
# capacity = total_segments * segment_length

# n_test_keys = 1_000_000

# @cuda.jit
# def cu_contain(results: Keys, fingerprints: Keys, keys: Keys):
#     tid = cuda.grid(1)
#     if tid < n_test_keys:
#         h0 = (keys[tid] >> 32) * segment_count_length + ((keys[tid] & 0xFFFFFFFF) * segment_count_length >> 32) >> 32
#         h1 = h0 + segment_length ^ (keys[tid] >> 18) & segment_length_mask
#         h2 = h0 + 2 * segment_length ^ keys[tid] & segment_length_mask
#         results[tid] = (keys[tid] ^ fingerprints[h0] ^ fingerprints[h1] ^ fingerprints[h2]) == 0

# F = cuda.to_device(fingerprints)
# R = cuda.to_device(np.zeros(n_test_keys, dtype=Key))
# np.random.seed(0)
# K = cuda.to_device(np.random.randint(0, np.iinfo(Key).max, n_test_keys, dtype=Key))

# cu_contain.forall(size)(R, F, K)
# R.copy_to_host().sum()

In [ ]:
# %timeit cu_contain.forall(size)(R, F, K)
# # 894 µs ± 69.4 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)